# ROPG-KD Data Generation
Generates `data/ropg_kd/{train,val}.jsonl` — scored (query, persona, top-K chunks) triples used to train the ROPG-KD retriever.

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 × 1 is enough).
2. Enable internet access.
3. Add Kaggle secrets: `OPENAI_API_KEY` and `OPENAI_BASE_URL`.
4. Attach the `simurgh-data` dataset (contains `chunks/corpus.jsonl`, `questions/`, `splits/`).

In [1]:
!pip install -q sentence-transformers openai numpy

## Config

In [2]:
import os

# ── Kaggle secrets ──────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient

    _s = UserSecretsClient()
    OPENAI_API_KEY = "aa-X78dmVEH65Nu2WaynoIcCiVbAY3PF7POt7XKRAa181S4AdGU"
    OPENAI_BASE_URL = "https://api.avalai.ir/v1"
except Exception:
    # fallback: set these in the environment before running locally
    OPENAI_API_KEY = "aa-X78dmVEH65Nu2WaynoIcCiVbAY3PF7POt7XKRAa181S4AdGU"
    OPENAI_BASE_URL = "https://api.avalai.ir/v1"

# ── Paths ────────────────────────────────────────────────────────────────────
# Change DATASET_SLUG to match the name you gave your Kaggle dataset.
DATASET_SLUG = "simurgh-data"
DATA_ROOT = "."
OUTPUT_DIR = "./ropg_kd"

# ── Inline config (mirrors configs/datagen_ropg.yaml) ───────────────────────
CFG = {
    "embedder": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "device": "cuda",  # Kaggle GPU
        "batch_size": 4,  # T4 has 16 GB; decoder attention is memory-hungry
        "fp16": True,  # halves VRAM; Qwen3-Embedding supports it
    },
    "retriever": {"top_k": 20},
    "judge": {
        "model": "gpt-4o-mini",
        "temperature": 0.0,
        "max_completion_tokens": 16000,
    },
    "data": {
        "chunks": f"{DATA_ROOT}/chunks/corpus.jsonl",
        "questions_dir": f"{DATA_ROOT}/questions",
        "splits_dir": f"{DATA_ROOT}/splits",
        "output_dir": OUTPUT_DIR,
    },
    "seed": 42,
}

## Core classes

In [3]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


class Qwen3Embedder:
    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-Embedding-0.6B",
        device: str = "cuda",
        batch_size: int = 4,
        fp16: bool = True,
    ) -> None:
        model_kwargs = {"torch_dtype": torch.float16} if fp16 else {}
        self.model = SentenceTransformer(
            model_name, device=device, trust_remote_code=True, model_kwargs=model_kwargs
        )
        self.batch_size = batch_size
        self.dim: int = self.model.get_embedding_dimension()

    def encode(self, texts: list) -> np.ndarray:
        vecs = self.model.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        return np.array(vecs, dtype=np.float32)

    def encode_query(self, texts: list, instruction: str = "") -> np.ndarray:
        if instruction:
            prompt = f"Instruct: {instruction}\nQuery: "
            vecs = self.model.encode(
                texts,
                prompt=prompt,
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        else:
            vecs = self.model.encode(
                texts,
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        return np.array(vecs, dtype=np.float32)

In [4]:
import openai


class OpenAICompatClient:
    def __init__(self, base_url, api_key, model, temperature=0.0, max_completion_tokens=16):
        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_completion_tokens = max_completion_tokens

    def chat(self, messages: list) -> str:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_completion_tokens=self.max_completion_tokens,
        )
        return resp.choices[0].message.content or ""

In [5]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

## Helper functions

In [6]:
import json
import logging
import re
from pathlib import Path

from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

FLOAT_RE = re.compile(r"[\d.]+")

JUDGE_SYSTEM = "You are an expert Persian language tutor evaluating study materials."


def load_corpus(path: Path):
    chunk_ids, texts = [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        chunk_ids.append(rec["chunk_id"])
        texts.append(rec["text"])
    if not texts:
        raise ValueError(f"Corpus file {path} is empty")
    return chunk_ids, texts


def parse_float_score(response: str) -> float:
    m = FLOAT_RE.search(response)
    if m is None:
        logger.warning("Could not parse score from: %r", response)
        return 0.0
    return max(0.0, min(1.0, float(m.group())))


def build_judge_messages(query: str, persona_rendered: str, chunk_text: str):
    user = (
        "A student with the following profile is trying to answer an exam question:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Exam question: {query}\n\n"
        f"Candidate study passage:\n{chunk_text}\n\n"
        "Rate 0.0–1.0 how useful this passage is for helping this specific student answer "
        "the question. Consider:\n"
        "  - Does the depth match the student's comprehension level?\n"
        "  - Does it provide what this student needs (simple paraphrase vs. deep analysis)?\n"
        "  - Is the style appropriate (hand-holding vs. terse treatment)?\n\n"
        "Respond with a single decimal number only, e.g. 0.73"
    )
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]


def score_chunks(judge, query, persona_rendered, chunk_ids, chunk_texts):
    docs = []
    for cid, ctext in zip(chunk_ids, chunk_texts, strict=True):
        msgs = build_judge_messages(query, persona_rendered, ctext)
        score = parse_float_score(judge.chat(msgs))
        docs.append({"chunk_id": int(cid), "text": ctext, "teacher_score": score})
    return docs


def load_split_qids(split_path: Path):
    entries = []
    for raw in split_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        exam_stem, qid = line.split(":", 1)
        entries.append((exam_stem.strip(), qid.strip(), line))
    return entries


def load_question(exam_stem: str, qid: str, questions_dir: Path) -> str:
    qfile = questions_dir / f"{exam_stem}.json"
    data = json.loads(qfile.read_text(encoding="utf-8"))
    for q in data.get("questions", []):
        if q["id"] == qid:
            return q["stem"]
    raise KeyError(f"Question {qid!r} not found in {qfile}")


def retrieve_top_k(query_vec, chunk_matrix, top_k, chunk_ids, chunk_texts):
    sims = query_vec.squeeze() @ chunk_matrix.T
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [chunk_ids[i] for i in top_idx], [chunk_texts[i] for i in top_idx]

## Run pipeline

In [7]:
np.random.seed(CFG["seed"])

corpus_path = Path(CFG["data"]["chunks"])
questions_dir = Path(CFG["data"]["questions_dir"])
splits_dir = Path(CFG["data"]["splits_dir"])
output_dir = Path(CFG["data"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

logger.info("Loading corpus from %s", corpus_path)
chunk_ids, chunk_texts = load_corpus(corpus_path)
logger.info("%d chunks loaded", len(chunk_ids))

20:28:46 [INFO] Loading corpus from chunks/corpus.jsonl
20:28:46 [INFO] 171 chunks loaded


In [8]:
logger.info("Encoding corpus with %s on %s …", CFG["embedder"]["model"], CFG["embedder"]["device"])
embedder = Qwen3Embedder(
    model_name=CFG["embedder"]["model"],
    device=CFG["embedder"]["device"],
    batch_size=CFG["embedder"]["batch_size"],
    fp16=CFG["embedder"]["fp16"],
)
chunk_matrix = embedder.encode(chunk_texts)
logger.info("Corpus matrix shape: %s", chunk_matrix.shape)

20:28:46 [INFO] Encoding corpus with Qwen/Qwen3-Embedding-0.6B on cuda …
20:28:47 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
20:28:47 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
20:28:47 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/modules.json "HTTP/1.1 200 OK"
20:28:47 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
20:28:47 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/config_sentence_transformers.json "HTTP/1.1 200 OK"
20:28:47 [INFO] Loading SentenceTransformer model from Qwen/Qwen3-Embedding-0.

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

20:28:51 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
20:28:51 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
20:28:51 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
20:28:51 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
20:28:51 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
20:28:52 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/tokenizer_config.json "HTTP/1.1 200 OK"
20:28:52 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwe

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

20:29:00 [INFO] Corpus matrix shape: (171, 1024)


In [9]:
# judge = OpenAICompatClient(
#     base_url=OPENAI_BASE_URL,
#     api_key=OPENAI_API_KEY,
#     model=CFG["judge"]["model"],
#     temperature=CFG["judge"]["temperature"],
#     max_completion_tokens=CFG["judge"]["max_completion_tokens"],
# )

# top_k = CFG["retriever"]["top_k"]
# train_profiles = train_personas()

# for split in ("train", "val"):
#     split_path = splits_dir / f"{split}_qids.txt"
#     if not split_path.exists():
#         logger.warning("Split file not found: %s — skipping", split_path)
#         continue

#     output_path = output_dir / f"{split}.jsonl"
#     entries = load_split_qids(split_path)
#     logger.info("Processing %s split (%d questions) → %s", split, len(entries), output_path)

#     with output_path.open("w", encoding="utf-8") as fh:
#         for exam_stem, qid, raw_line in tqdm(entries, desc=split, unit="q"):
#             try:
#                 query = load_question(exam_stem, qid, questions_dir)
#             except (FileNotFoundError, KeyError) as exc:
#                 logger.warning("Skipping %s: %s", raw_line, exc)
#                 continue

#             for persona in tqdm(train_profiles, desc="  personas", leave=False, unit="p"):
#                 persona_rendered = render_profile(persona.id)
#                 query_vec = embedder.encode_query([query], instruction=persona_rendered)
#                 top_ids, top_texts = retrieve_top_k(
#                     query_vec, chunk_matrix, top_k, chunk_ids, chunk_texts
#                 )
#                 docs = score_chunks(judge, query, persona_rendered, top_ids, top_texts)
#                 rec = {"query": query, "persona_id": persona.id, "docs": docs}
#                 fh.write(json.dumps(rec, ensure_ascii=False) + "\n")


#     logger.info("Wrote %s", output_path)

# logger.info("All splits complete. Output in %s", output_dir)

In [10]:
# from concurrent.futures import ThreadPoolExecutor, as_completed

# np.random.seed(CFG["seed"])

# corpus_path = Path(CFG["data"]["chunks"])
# questions_dir = Path(CFG["data"]["questions_dir"])
# splits_dir = Path(CFG["data"]["splits_dir"])
# output_dir = Path(CFG["data"]["output_dir"])
# output_dir.mkdir(parents=True, exist_ok=True)

# logger.info("Loading corpus from %s", corpus_path)
# chunk_ids, chunk_texts = load_corpus(corpus_path)
# logger.info("%d chunks loaded", len(chunk_ids))

# # ── Embed ────────────────────────────────────────────────────────────────────
# logger.info("Encoding corpus with %s on %s …", CFG["embedder"]["model"], CFG["embedder"]["device"])
# embedder = Qwen3Embedder(
#     model_name=CFG["embedder"]["model"],
#     device=CFG["embedder"]["device"],
#     batch_size=CFG["embedder"]["batch_size"]*3,
#     fp16=CFG["embedder"]["fp16"],
# )
# chunk_matrix = embedder.encode(chunk_texts)
# logger.info("Corpus matrix shape: %s", chunk_matrix.shape)

# # ── Judge & config ──────────────────────────────────────────────────────────
# judge = OpenAICompatClient(
#     base_url=OPENAI_BASE_URL,
#     api_key=OPENAI_API_KEY,
#     model=CFG["judge"]["model"],
#     temperature=CFG["judge"]["temperature"],
#     max_completion_tokens=CFG["judge"]["max_completion_tokens"],
# )

# top_k = CFG["retriever"]["top_k"]
# train_profiles = train_personas()

# # ── Multithreaded scoring ──────────────────────────────────────────────────
# MAX_WORKERS = 4  # adjust based on API rate limits and network bandwidth

# def score_single(args, query, persona_rendered, judge_client):
#     """Score one chunk with the judge. Called from multiple threads."""
#     cid, ctext = args
#     try:
#         msgs = build_judge_messages(query, persona_rendered, ctext)
#         score = parse_float_score(judge_client.chat(msgs))
#         return {"chunk_id": cid, "text": ctext, "teacher_score": score}
#     except Exception as e:
#         logger.warning("Scoring failed for chunk %s: %s", cid, e)
#         return {"chunk_id": cid, "text": ctext, "teacher_score": 0.0}

# for split in ("train", "val"):
#     split_path = splits_dir / f"{split}_qids.txt"
#     if not split_path.exists():
#         logger.warning("Split file not found: %s — skipping", split_path)
#         continue

#     output_path = output_dir / f"{split}.jsonl"
#     entries = load_split_qids(split_path)
#     logger.info("Processing %s split (%d questions) → %s", split, len(entries), output_path)

#     with output_path.open("w", encoding="utf-8") as fh:
#         for exam_stem, qid, raw_line in tqdm(entries, desc=split, unit="q"):
#             try:
#                 query = load_question(exam_stem, qid, questions_dir)
#             except (FileNotFoundError, KeyError) as exc:
#                 logger.warning("Skipping %s: %s", raw_line, exc)
#                 continue

#             for persona in tqdm(train_profiles, desc="  personas", leave=False, unit="p"):
#                 persona_rendered = render_profile(persona.id)
#                 query_vec = embedder.encode_query([query], instruction=persona_rendered)
#                 top_ids, top_texts = retrieve_top_k(
#                     query_vec, chunk_matrix, top_k, chunk_ids, chunk_texts
#                 )

#                 # Parallelize scoring of the retrieved chunks
#                 args_list = list(zip(top_ids, top_texts))
#                 docs = [None] * len(args_list)

#                 with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#                     future_to_idx = {
#                         executor.submit(score_single, args, query, persona_rendered, judge): i
#                         for i, args in enumerate(args_list)
#                     }
#                     # Progress bar for chunk scoring
#                     for future in tqdm(
#                         as_completed(future_to_idx),
#                         total=len(args_list),
#                         desc="    scoring chunks",
#                         leave=False,
#                         unit="chunk",
#                     ):
#                         idx = future_to_idx[future]
#                         docs[idx] = future.result()

#                 # Write the record for this (query, persona)
#                 rec = {"query": query, "persona_id": persona.id, "docs": docs}
#                 fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

#     logger.info("Wrote %s", output_path)

# logger.info("All splits complete. Output in %s", output_dir)

In [11]:
!pip install faiss-gpu

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [12]:
# import json
# import logging
# from pathlib import Path
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from collections import defaultdict
# import numpy as np
# import faiss
# from tqdm import tqdm

# import faiss
# import numpy as np

# # Ensure chunk_matrix is float32 and L2-normalized for cosine similarity
# chunk_matrix = chunk_matrix.astype(np.float32)
# faiss.normalize_L2(chunk_matrix)   # in-place normalization

# # Build Faiss index (Inner Product = cosine after normalization)
# dim = chunk_matrix.shape[1]
# index = faiss.IndexFlatIP(dim)      # IP = inner product
# index.add(chunk_matrix)

# logger.info("Faiss index built with %d vectors, dimension %d", index.ntotal, dim)


# MAX_WORKERS = 10

# # ── Judge & config ──────────────────────────────────────────────────────────
# judge = OpenAICompatClient(
#     base_url=OPENAI_BASE_URL,
#     api_key=OPENAI_API_KEY,
#     model=CFG["judge"]["model"],
#     temperature=CFG["judge"]["temperature"],
#     max_completion_tokens=CFG["judge"]["max_completion_tokens"],
# )

# top_k = CFG["retriever"]["top_k"]
# train_profiles = train_personas()

# # ── Suppress verbose HTTP logs from successful requests ──────────────
# for lib in ["httpx", "urllib3", "openai", "httpcore"]:
#     logging.getLogger(lib).setLevel(logging.WARNING)

# # (Assume logger, embedder, CFG, train_profiles, chunk_texts, etc. are defined)

# # ── Helper: score one triple, now returns pair_idx ────────────────────
# def score_triple(query, persona_rendered, doc_text, persona_id, chunk_id, pair_idx):
#     """Score one (query, persona, chunk) with the judge."""
#     try:
#         msgs = build_judge_messages(query, persona_rendered, doc_text)
#         score = parse_float_score(judge.chat(msgs))
#         return {
#             "query": query,
#             "persona_id": persona_id,
#             "chunk_id": chunk_id,
#             "text": doc_text,
#             "teacher_score": score,
#             "pair_idx": pair_idx
#         }
#     except Exception as e:
#         logger.warning("Scoring failed for chunk %s (persona %s): %s",
#                        chunk_id, persona_id, e)
#         return {
#             "query": query,
#             "persona_id": persona_id,
#             "chunk_id": chunk_id,
#             "text": doc_text,
#             "teacher_score": 0.0,
#             "pair_idx": pair_idx
#         }

# # ── Process splits ──────────────────────────────────────────────────────
# for split in ("train", "val"):
#     split_path = splits_dir / f"{split}_qids.txt"
#     if not split_path.exists():
#         logger.warning("Split file not found: %s — skipping", split_path)
#         continue

#     output_path = output_dir / f"{split}.jsonl"
#     entries = load_split_qids(split_path)

#     # ── Resumption: read already processed pairs ──────────────────────
#     done_pairs = set()
#     if output_path.exists():
#         with output_path.open("r", encoding="utf-8") as fh:
#             for line in fh:
#                 line = line.strip()
#                 if not line:
#                     continue
#                 try:
#                     rec = json.loads(line)
#                     key = (rec["query"], rec["persona_id"])
#                     done_pairs.add(key)
#                 except json.JSONDecodeError:
#                     logger.warning("Skipping malformed line in %s", output_path)
#         logger.info("Found %d already processed pairs in %s, skipping them.",
#                     len(done_pairs), output_path)

#     logger.info("Processing %s split (%d questions) → %s", split, len(entries), output_path)

#     # 1. Build list of (query, persona_rendered, persona_id) filtering done pairs
#     pair_list = []
#     for exam_stem, qid, raw_line in entries:
#         try:
#             query = load_question(exam_stem, qid, questions_dir)
#         except (FileNotFoundError, KeyError) as exc:
#             logger.warning("Skipping %s: %s", raw_line, exc)
#             continue
#         for persona in train_profiles:
#             key = (query, persona.id)
#             if key in done_pairs:
#                 continue   # skip already completed
#             persona_rendered = render_profile(persona.id)
#             pair_list.append((query, persona_rendered, persona.id))

#     if not pair_list:
#         logger.info("No new pairs to process for split %s", split)
#         continue

#     # 2. Batch encode all query–persona pairs
#     logger.info("Encoding %d new query–persona pairs ...", len(pair_list))
#     texts_to_encode = [f"{inst}\n{q}" for q, inst, _ in pair_list]
#     query_vectors = embedder.encode(texts_to_encode)
#     query_vectors = query_vectors.astype(np.float32)
#     faiss.normalize_L2(query_vectors)

#     # 3. Batch retrieval
#     logger.info("Retrieving top-%d chunks for %d pairs ...", top_k, len(pair_list))
#     distances, indices = index.search(query_vectors, top_k)

#     # 4. Prepare for incremental scoring
#     pair_pending = {}
#     pair_scores = defaultdict(list)

#     # Open output file in append mode (will create if not exists)
#     with output_path.open("a", encoding="utf-8") as out_fh:
#         # Submit all scoring tasks
#         logger.info("Submitting scoring tasks ...")
#         with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#             future_to_pair = {}
#             for pair_idx, (query, inst, p_id) in enumerate(pair_list):
#                 chunk_ids = indices[pair_idx]
#                 valid_chunk_ids = [cid for cid in chunk_ids if cid != -1]
#                 pair_pending[pair_idx] = len(valid_chunk_ids)
#                 for cid in valid_chunk_ids:
#                     doc_text = chunk_texts[cid]
#                     future = executor.submit(
#                         score_triple, query, inst, doc_text, p_id, cid, pair_idx
#                     )
#                     future_to_pair[future] = pair_idx

#             logger.info("Scoring %d triples with %d workers ...", len(future_to_pair), MAX_WORKERS)
#             with tqdm(total=len(future_to_pair), desc=f"Scoring {split}", unit="triple") as pbar:
#                 for future in as_completed(future_to_pair):
#                     pair_idx = future_to_pair[future]
#                     try:
#                         res = future.result()
#                         pair_scores[pair_idx].append(res)
#                     except Exception as e:
#                         logger.error("Unexpected error in scoring thread: %s", e)
#                     finally:
#                         pair_pending[pair_idx] -= 1
#                         # If this pair is complete, write its record
#                         if pair_pending[pair_idx] == 0:
#                             query, inst, p_id = pair_list[pair_idx]
#                             # Rebuild docs in the original retrieval order
#                             score_map = {s["chunk_id"]: s for s in pair_scores[pair_idx]}
#                             docs = []
#                             for cid in indices[pair_idx]:
#                                 if cid == -1:
#                                     continue
#                                 s = score_map.get(cid)
#                                 if s:
#                                     docs.append({
#                                         "chunk_id": s["chunk_id"],
#                                         "text": s["text"],
#                                         "teacher_score": s["teacher_score"]
#                                     })
#                                 else:
#                                     # Fallback (should not happen)
#                                     docs.append({
#                                         "chunk_id": cid,
#                                         "text": chunk_texts[cid],
#                                         "teacher_score": 0.0
#                                     })
#                             rec = {"query": query, "persona_id": p_id, "docs": docs}
#                             out_fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
#                             out_fh.flush()   # ensure it's written immediately
#                             # Free memory for this pair
#                             del pair_scores[pair_idx]
#                             del pair_pending[pair_idx]
#                         pbar.update(1)

#     logger.info("Appended new results to %s", output_path)

# logger.info("All splits complete. Output in %s", output_dir)

In [13]:
import json
import logging
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import numpy as np
import faiss
from tqdm import tqdm

# ---------- existing imports and setup (unchanged) ----------
# Ensure chunk_matrix is float32 and L2-normalized for cosine similarity
chunk_matrix = chunk_matrix.astype(np.float32)
faiss.normalize_L2(chunk_matrix)   # in-place normalization

# Build Faiss index (Inner Product = cosine after normalization)
dim = chunk_matrix.shape[1]
index = faiss.IndexFlatIP(dim)      # IP = inner product
index.add(chunk_matrix)

logger.info("Faiss index built with %d vectors, dimension %d", index.ntotal, dim)

MAX_WORKERS = 7

# ── Judge & config ──────────────────────────────────────────────────────────
judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=CFG["judge"]["model"],
    temperature=CFG["judge"]["temperature"],
    max_completion_tokens=CFG["judge"]["max_completion_tokens"],
)

top_k = CFG["retriever"]["top_k"]
train_profiles = train_personas()

# ---------- helpers (unchanged) ----------
FLOAT_RE = re.compile(r"[\d.]+")
JUDGE_SYSTEM = "You are an expert Persian language tutor evaluating study materials."

# ── Suppress verbose HTTP logs from successful requests ──────────────
for lib in ["httpx", "urllib3", "openai", "httpcore"]:
    logging.getLogger(lib).setLevel(logging.WARNING)

# (Assume logger, embedder, CFG, train_profiles, chunk_texts, etc. are defined)

def parse_float_score(response: str) -> float:
    m = FLOAT_RE.search(response)
    if m is None:
        logger.warning("Could not parse score from: %r", response)
        return 0.0
    return max(0.0, min(1.0, float(m.group())))

def build_judge_messages(query: str, persona_rendered: str, chunk_text: str):
    user = (
        "A student with the following profile is trying to answer an exam question:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Exam question: {query}\n\n"
        f"Candidate study passage:\n{chunk_text}\n\n"
        "Rate 0.0–1.0 how useful this passage is for helping this specific student answer "
        "the question. Consider:\n"
        "  - Does the depth match the student's comprehension level?\n"
        "  - Does it provide what this student needs (simple paraphrase vs. deep analysis)?\n"
        "  - Is the style appropriate (hand-holding vs. terse treatment)?\n\n"
        "Respond with a single decimal number only, e.g. 0.73"
    )
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]

# ---------- modified batched scoring functions ----------
def build_batched_judge_messages(query: str, persona_rendered: str,
                                 chunk_list: list[tuple[int, str]]) -> list[dict]:
    """
    chunk_list: list of (chunk_id, text)
    Builds a prompt that lists all passages with [1], [2], ... and asks for
    a list of scores in the same order.
    """
    num = len(chunk_list)
    passages = "\n".join(f"[{i+1}] {text}" for i, (_, text) in enumerate(chunk_list))
    user = (
        "A student with the following profile is trying to answer an exam question:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Exam question: {query}\n\n"
        "Below are several candidate study passages, each numbered.\n"
        f"{passages}\n\n"
        "For each passage, rate 0.0–1.0 how useful it is for helping this specific student answer "
        "the question. Consider depth, comprehension level, and style.\n"
        f"Respond with a JSON object like {{\"scores\": [0.5, 0.8, ...]}} containing **exactly {num}** scores "
        "in the same order as the passages above. Provide only the JSON, no extra text."
    )
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]

def parse_batched_scores(response: str, num_chunks: int) -> list[float]:
    """
    Extract a list of floats from the response. Expected JSON: {"scores": [...]}
    If the list length differs from num_chunks, pad missing entries with 0.0 (or truncate).
    Falls back to parsing comma-separated numbers if JSON fails, also padding as needed.
    """
    def _sanitize(val):
        return max(0.0, min(1.0, float(val)))

    # Try JSON first
    try:
        data = json.loads(response)
        scores = data.get("scores")
        if isinstance(scores, list):
            parsed = [_sanitize(x) for x in scores]
            if len(parsed) == num_chunks:
                return parsed
            elif len(parsed) < num_chunks:
                logger.warning(
                    "Received %d scores but expected %d. Padding with zeros.",
                    len(parsed), num_chunks
                )
                parsed.extend([0.0] * (num_chunks - len(parsed)))
                return parsed
            else:  # more scores than needed
                logger.warning(
                    "Received %d scores but expected only %d. Truncating.",
                    len(parsed), num_chunks
                )
                return parsed[:num_chunks]
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    # Fallback: find all numbers in the string
    nums = list(map(float, re.findall(r"[\d.]+", response)))
    if len(nums) >= num_chunks:
        return [_sanitize(x) for x in nums[:num_chunks]]
    elif nums:
        logger.warning(
            "Fallback found only %d numbers, expected %d. Padding with zeros.",
            len(nums), num_chunks
        )
        scores = [_sanitize(x) for x in nums]
        scores.extend([0.0] * (num_chunks - len(scores)))
        return scores
    else:
        logger.warning("Could not parse any scores from response: %r", response)
        return [0.0] * num_chunks

# ---------- batched scoring function (unchanged except the call) ----------
def score_pair(judge, query: str, persona_rendered: str, persona_id: str,
               chunk_ids: list[int], chunk_texts: dict[int, str]) -> dict:
    """
    Score all chunks for a single (query, persona) in one LLM call.
    chunk_ids: list of chunk IDs (length = top_k)
    Returns a record dict with 'query', 'persona_id', and 'docs' (list of {chunk_id, text, teacher_score})
    """
    # Build list of (chunk_id, text)
    chunk_list = [(cid, chunk_texts[cid]) for cid in chunk_ids if cid != -1]
    if not chunk_list:
        # no valid chunks – return zeros
        return {
            "query": query,
            "persona_id": persona_id,
            "docs": [{"chunk_id": int(cid), "text": chunk_texts[int(cid)], "teacher_score": 0.0}
                     for cid in chunk_ids if cid != -1]
        }
    msgs = build_batched_judge_messages(query, persona_rendered, chunk_list)
    try:
        response = judge.chat(msgs)
        # logger.warning(response)
        scores = parse_batched_scores(response, len(chunk_list))
    except Exception as e:
        logger.warning("Batched scoring failed for persona %s: %s", persona_id, e)
        scores = [0.0] * len(chunk_list)

    docs = []
    for (cid, text), score in zip(chunk_list, scores):
        docs.append({
            "chunk_id": int(cid),
            "text": text,
            "teacher_score": score
        })
    return {"query": query, "persona_id": persona_id, "docs": docs}

# ---------- main processing loop (unchanged) ----------
for split in ("train", "val"):
    split_path = splits_dir / f"{split}_qids.txt"
    if not split_path.exists():
        logger.warning("Split file not found: %s — skipping", split_path)
        continue

    output_path = output_dir / f"{split}.jsonl"
    entries = load_split_qids(split_path)

    # ---- resumption ----
    done_pairs = set()
    if output_path.exists():
        with output_path.open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    done_pairs.add((rec["query"], rec["persona_id"]))
                except json.JSONDecodeError:
                    logger.warning("Skipping malformed line in %s", output_path)
        logger.info("Found %d already processed pairs in %s, skipping them.",
                    len(done_pairs), output_path)

    logger.info("Processing %s split (%d questions) → %s", split, len(entries), output_path)

    # 1. Build list of (query, persona_rendered, persona_id) filtering done pairs
    pair_list = []
    for exam_stem, qid, raw_line in entries:
        try:
            query = load_question(exam_stem, qid, questions_dir)
        except (FileNotFoundError, KeyError) as exc:
            logger.warning("Skipping %s: %s", raw_line, exc)
            continue
        for persona in train_profiles:
            key = (query, persona.id)
            if key in done_pairs:
                continue
            persona_rendered = render_profile(persona.id)
            pair_list.append((query, persona_rendered, persona.id))

    if not pair_list:
        logger.info("No new pairs to process for split %s", split)
        continue

    # 2. Batch encode all query–persona pairs
    logger.info("Encoding %d new query–persona pairs ...", len(pair_list))
    texts_to_encode = [f"{inst}\n{q}" for q, inst, _ in pair_list]
    query_vectors = embedder.encode(texts_to_encode)
    query_vectors = query_vectors.astype(np.float32)
    faiss.normalize_L2(query_vectors)

    # 3. Batch retrieval
    logger.info("Retrieving top-%d chunks for %d pairs ...", top_k, len(pair_list))
    distances, indices = index.search(query_vectors, top_k)

    # 4. Submit one scoring task per pair (batched over all chunks)
    logger.info("Submitting batched scoring tasks for %d pairs ...", len(pair_list))
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_pair = {}
        for pair_idx, (query, inst, p_id) in enumerate(pair_list):
            chunk_ids = indices[pair_idx]
            valid_chunk_ids = [cid for cid in chunk_ids if cid != -1]
            if not valid_chunk_ids:
                # no chunks retrieved – write a zero record
                rec = {
                    "query": query,
                    "persona_id": p_id,
                    "docs": []
                }
                with output_path.open("a", encoding="utf-8") as out_fh:
                    out_fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue
            future = executor.submit(
                score_pair, judge, query, inst, p_id, valid_chunk_ids, chunk_texts
            )
            future_to_pair[future] = pair_idx

        logger.info("Scoring %d pairs with %d workers ...", len(future_to_pair), MAX_WORKERS)
        with tqdm(total=len(future_to_pair), desc=f"Scoring {split}", unit="pair") as pbar:
            for future in as_completed(future_to_pair):
                pair_idx = future_to_pair[future]
                try:
                    rec = future.result()
                    # write record immediately
                    with output_path.open("a", encoding="utf-8") as out_fh:
                        out_fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        out_fh.flush()
                    pbar.update(1)
                except Exception as e:
                    logger.error("Unexpected error in scoring thread for pair %d: %s", pair_idx, e)
                    # fallback: write zeros
                    query, inst, p_id = pair_list[pair_idx]
                    chunk_ids = indices[pair_idx]
                    docs = [{"chunk_id": int(cid), "text": chunk_texts[int(cid)], "teacher_score": 0.0}
                            for cid in chunk_ids if cid != -1]
                    rec = {"query": query, "persona_id": p_id, "docs": docs}
                    with output_path.open("a", encoding="utf-8") as out_fh:
                        out_fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    pbar.update(1)
                finally:
                    pbar.update(1)

    logger.info("Appended new results to %s", output_path)

logger.info("All splits complete. Output in %s", output_dir)

20:29:02 [INFO] Loading faiss with AVX512 support.
20:29:02 [INFO] Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
20:29:02 [INFO] Loading faiss with AVX2 support.
20:29:02 [INFO] Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
20:29:02 [INFO] Loading faiss.
20:29:02 [INFO] Successfully loaded faiss.
20:29:02 [INFO] Faiss index built with 171 vectors, dimension 1024
20:29:02 [INFO] Found 52 already processed pairs in ropg_kd/train.jsonl, skipping them.
20:29:02 [INFO] Processing train split (554 questions) → ropg_kd/train.jsonl
20:29:04 [INFO] Encoding 1610 new query–persona pairs ...


Batches:   0%|          | 0/403 [00:00<?, ?it/s]

20:29:20 [INFO] Retrieving top-20 chunks for 1610 pairs ...
20:29:20 [INFO] Submitting batched scoring tasks for 1610 pairs ...
20:29:20 [INFO] Scoring 1610 pairs with 7 workers ...
20:30:49 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:33:40 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:35:33 [WARNING] Received 6 scores but expected 20. Padding with zeros.
20:36:40 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:36:41 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:40:39 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:40:43 [WARNING] Received 6 scores but expected 20. Padding with zeros.
20:41:31 [WARNING] Received 8 scores but expected 20. Padding with zeros.
20:43:40 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:43:44 [WARNING] Received 5 scores but expected 20. Padding with zeros.
20:43:54 [WARNING] Received 6 scores but expected 20. Padding with zeros.
20:4

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

20:45:48 [INFO] Retrieving top-20 chunks for 48 pairs ...
20:45:48 [INFO] Submitting batched scoring tasks for 48 pairs ...
20:45:49 [INFO] Scoring 48 pairs with 7 workers ...
Scoring val: 96pair [00:28,  3.39pair/s]          
20:46:17 [INFO] Appended new results to ropg_kd/val.jsonl
20:46:17 [INFO] All splits complete. Output in ropg_kd
